# Quick LLM tests

Manual side by side comparison of the three models on a few questions. Gives an intuition before running the full benchmark in `02_experiments`.

In [ ]:
import sys, gc, torch
sys.path.insert(0, '..')

from src.embedding import Embedder
from src.vectorstore import VectorStore
from src.generation import LLMGenerator
from src.rag_pipeline import RAGPipeline

In [ ]:
# embedder + chromadb loaded once
emb = Embedder('OrdalieTech/Solon-embeddings-base-0.1')
vs = VectorStore('../data/chroma_db', 'collection_solon_base')
print('chunks:', vs.count())

In [ ]:
# test questions (one per source type, written in French because the corpus is French)
questions = [
    "Dois-je nommer un DPO si j'ai 30 salaries ?",
    "Combien de temps puis-je conserver un CV de candidat non retenu ?",
    "Pourquoi Google a ete sanctionne en 2019 ?",
]

## Loop over the three models

Smallest to largest. Memory is released between each to avoid saturation.

In [ ]:
def free_mem():
    gc.collect()
    if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        torch.mps.empty_cache()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

models = [
    'TinyLlama/TinyLlama-1.1B-Chat-v1.0',
    'Qwen/Qwen2.5-1.5B-Instruct',
    'Qwen/Qwen2.5-3B-Instruct',
]

llm = None
for m in models:
    print('=' * 60)
    print('model:', m)
    if llm is not None:
        del llm
    free_mem()
    llm = LLMGenerator(m)
    pipe = RAGPipeline(emb, vs, llm, prompt_name='citation')
    for q in questions:
        print()
        print('Q:', q)
        resp = pipe.answer(q)
        print('A:', resp.answer[:600])
        print('  total latency:', resp.latency_ms['total'], 'ms')

## Observations

Fill in manually after reading the responses:
- TinyLlama: ...
- Qwen 1.5B: ...
- Qwen 3B: ...